In [18]:
from cobra.io import load_model, load_json_model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

In [19]:
# Contains the main steps of the BayesOpt
%run BayesOpt_MOBO.ipynb

# Plotting functions to be used across notebooks
%run Plotting_MOBO.ipynb

# contains helper functions 
%run HelperFunctions_MOBO.ipynb

# Function Definitions

In [20]:
def generate_samples_from_multivariate_gaussian(
        data, 
        variance = 0.05, 
        sample_size = 1000, 
        dec_var_names = None, 
        bounds = None):
    
    """
    Generate samples from a multivariate Gaussian distribution for a single medium composition.
    Only valid samples, i.e. such for which all medium components are within the bounds that were set during the optimisation are considered.
    
    PARAMETERS:
    * data - dataframe - Dataframe containing the 
    * variance - float - Variance to be used for the multivariate Gaussian distribution, expressed as a percentage of the mean (default is 0.05 for 5%).
    * sample_size - integer - Number of samples to be drawn from the multivariate Gaussian distribution (default is 1000).
    * dec_var_names - list - List of decision variable names corresponding to the columns in data_test.
    * bounds - dictionary - the bounds for the medium components

    RETURNS:
    * samples_dict - dict - Dictionary where keys are row indices and values are arrays of samples drawn from the multivariate Gaussian distribution.

    
    COMMENT:
    A small number was added to the diagonal of the covariance matrix for regularization purposes, to ensure that it is positive definite and to avoid numerical issues during sampling. 
    This is a common practice when working with covariance matrices in multivariate distributions.
    """
    
    # generate an empty dataframe to add all samples
    samples_df = pd.DataFrame()
    
    # Generate multivariate Gaussian for each row
    for index, row in data.iterrows():

        medium = row.values
        #print(f"Generating samples for medium composition at index {index}: {medium}")
        
        # Set the variance to 5% of the mean for each variable
        covariance_matrix = np.diag((variance * medium) ** 2)
        
        # Regularization: Add a small value to the diagonal to ensure the covariance matrix is positive definite
        regularization_term = 1e-6
        covariance_matrix += np.eye(len(row)) * regularization_term
        
        # Create the multivariate normal distribution
        mvn = multivariate_normal(mean = medium, cov = covariance_matrix)
        
        # Samples: list of X samples drawn from the multivariate Gaussian distribution
        # Generate exactly sample_size non-negative samples with medium compounds inside the bounds defined  
        non_negative_samples = []
        while len(non_negative_samples) < sample_size:
            samples = mvn.rvs(size=sample_size)
            positive_samples = samples[np.all(samples >= 0, axis=1)]
            # Check if the samples are positive and within the bounds
            valid_samples = []
            for sample in positive_samples:
                is_valid = True
                for i, var_name in enumerate(dec_var_names):
                    lower_bound, upper_bound = bounds[var_name]
                    if not (lower_bound <= sample[i] <= upper_bound):
                        is_valid = False
                        break
                if is_valid:
                    valid_samples.append(sample)

            non_negative_samples.extend(valid_samples)
        
        # Trim to exactly sample_size samples
        non_negative_samples = np.array(non_negative_samples[:sample_size])

        # save in dataframe with columns corresponding to the decision variable names
        samples_per_medium_df = pd.DataFrame(non_negative_samples, columns=dec_var_names)
        # add new samples to the main dataframe
        samples_df = pd.concat([samples_df, samples_per_medium_df])
        # reset index of the main dataframe after adding new samples
        samples_df["index"] = list(range(len(samples_df.index)))
        samples_df.set_index("index",inplace=True)

    # convert the result to a dictionary for easier handling with model.medium (FBA step)   
    samples_dict = samples_df.to_dict("index")
    
    return samples_dict


In [21]:
def get_growth_and_cost(media_samples, MetModel = None, costs = None, production_rxn_id = None, traces = None):
    """
    Get growth and costs for every sample in media_samples.
    
    PARAMETERS:
    * media_samples - dict - containing different medium compositions, generate_samples_from_multivariate_gaussian function output
    * MetModel - COBRApy model - the metabolic model to be evaluated
    * costs - dictionary - cost for each medium component [£/mol]

    RETURNS:
    * growth_cand_cost_df - dataframe - with growth and cost for each sample in media_samples
    """
    
    samples_growth = []
    samples_costs = []
    samples_production = []

    for medium in media_samples.values():
        
        MetModel.medium = medium

        if traces is not None:
            # add trace metals to the medium composition 
            medium_with_traces = {**medium, **traces}
            # set current sample as the medium to the model
            MetModel.medium = medium_with_traces
        
        if costs is not None:
            # caclulate total cost
            cost_tot = sum(concentration * costs[key] for key, concentration in medium.items())
            samples_costs.append(cost_tot)
        
        '''FBA'''
        FBA_solution = MetModel.optimize() # run FBA
        # extract growth rate
        growth = FBA_solution.fluxes[biomass_rxn_id]
        samples_growth.append(growth)
        
        if production_rxn_id is not None:
            production = FBA_solution.fluxes[production_rxn_id]
            samples_production.append(production)        
        
    if costs is not None:
        growth_and_cost_df = pd.DataFrame({'growth_rate': samples_growth, 'cost': samples_costs})
        return growth_and_cost_df

    if production_rxn_id is not None:
        growth_and_production_df = pd.DataFrame({"growth_rate": samples_growth, "production": samples_production})
        return growth_and_production_df
    
    return samples_growth

# iML1515

### Load models, cost and trace metals from medium 

In [22]:
# load iML1515
model_iML1515 = load_model("iML1515")
print(model_iML1515)

iML1515


In [25]:
# M9 with essential trace metals
medium_iJO1366_reduced = {
    'EX_pi_e': 34.90, # in M9
    'EX_mn2_e': 0.001, # - required?; drops at 0.0001
    'EX_fe2_e': 0.1, # - required?; drops at 0.01
    'EX_glc__D_e': 10.0, # in M9
    'EX_zn2_e': 0.001, # - required?; drops at 0.0001
    'EX_mg2_e': 1.0, # in M9 
    'EX_ca2_e': 0.05, # in M9
    'EX_ni2_e': 0.001, # - required?; drops at 0.0001
    'EX_cu2_e': 0.001, # - required?; drops at 0.0001
    'EX_cobalt2_e': 0.0001, # - required; drops at 0.00001 
    'EX_mobd_e': 0.0005, # - required?; drops at 0.000001
    'EX_so4_e': 1.0, # in M9
    'EX_nh4_e': 9.3475, # in M9
    'EX_k_e': 11.02, # in M9
    'EX_na1_e': 52.038, # in M9
    'EX_cl_e': 13.6755, # in M9
    'EX_o2_e': 20.0, # in M9 II - drops at 10
}
bounds_iJO1366_reduced = { 
    'EX_pi_e': (0.0, 50),
    'EX_mn2_e': (0.001, 0.001), # fix "trace" 
    'EX_fe2_e': (0.1, 0.1), # fix "trace"
    'EX_glc__D_e': (1.0, 10),
    'EX_zn2_e': (0.001, 0.001), # fix "trace"
    'EX_mg2_e': (0.0, 10),
    'EX_ca2_e': (0.0, 10),
    'EX_ni2_e': (0.001, 0.001), # fix "trace"
    'EX_cu2_e': (0.001, 0.001), # fix "trace"
    'EX_cobalt2_e': (0.0001, 0.0001), # fix "trace"
    'EX_mobd_e': (0.0005, 0.0005), # fix "trace"
    'EX_so4_e': (0.0, 10),
    'EX_nh4_e': (0.0, 10),
    'EX_k_e': (0.0, 20),
    'EX_na1_e': (0.0, 100.0), # fix - can be set to 0
    'EX_cl_e': (0.0, 20),
    'EX_o2_e': (0, 20), # upper bound can't be set to 10 or lower
}
# costs are in £/mol
costs_iJO1366_reduced = {
    'EX_pi_e': 23.4234, # Phosphate - approximate (several sources)
    'EX_mn2_e': 0.0, #33.25, # Manganese - MnCl2·4H20
    'EX_fe2_e': 0.0, #37.5, # IronII - as iron sulfate FeSO4·7H2O
    'EX_glc__D_e': 7.7647236, # Glucose
    'EX_zn2_e': 0.0, #28.3, # Zinc - as Zn(CH3CHOOH)·H2O
    'EX_mg2_e': 19.1022, # Magnesium - as MgSO4·7H2O - approximate bc. half of 38.2044
    'EX_ca2_e': 18.08223, # Calcium - as CaCl2·2H2O
    'EX_ni2_e': 0.0, #53.24, # Nickel - as NiCl2·6H2O
    'EX_cu2_e': 0.0, #31.37, # Copper - CuCl2·2H2O
    'EX_cobalt2_e': 0.0, #114.39, # Cobalt - as CoCl2·6H2O
    'EX_mobd_e': 0.0, #184.12, # Molybdenum - molybdate NaMoO4·2H2O
    'EX_so4_e': 19.1022, # Sulfate - as MgSO4·7H2O - approximate bc. half of 38.2044
    'EX_nh4_e': 3.6748, # Ammonia - as NH4Cl - approximate (several sources and "side-effect"
    'EX_k_e': 20.82177, # Potassium - as KCl - approximate (several sources and "side-effect"
    'EX_na1_e': 0.0, # Sodium - as NaCl, Na2HPO4
    'EX_cl_e': 3.03888, # Chlorid - as NaCl, NH4Cl, CaCl2 - approximate price from NaCl
    'EX_o2_e': 0.0, # oxygen - no costs
}

# save traces for later use (FBA with different media combinations but keeping the trace metals fixed)
traces_iML1515 = { 
    'EX_mn2_e': 0.001, # fix "trace" 
    'EX_fe2_e': 0.1, # fix "trace"
    'EX_zn2_e': 0.001, # fix "trace"
    'EX_ni2_e': 0.001, # fix "trace"
    'EX_cu2_e': 0.001, # fix "trace"
    'EX_cobalt2_e': 0.0001, # fix "trace"
    'EX_mobd_e': 0.0005 # fix "trace"
}


In [26]:
file_name = "Results\\2026-03-31_BayesOpt_iML1515_growth-cost_200.json"
results_iML1515_200 = JSON_deserialize_load_results((file_name), model_iML1515)

In [27]:
# reaction ID for FBA objective function (biomass reaction)
biomass_rxn_id = "BIOMASS_Ec_iML1515_core_75p37M"

# bounds of biomass reaction
biomass_rxn = model_iML1515.reactions.get_by_id(biomass_rxn_id)
biomass_rxn.bounds = (0.0, 0.85)

# get initial values
model_iML1515.medium = medium_iJO1366_reduced

solution = model_iML1515.optimize()
init_growth = solution.fluxes[results_iML1515_200["biomass objective"]]
init_cost = calc_cost_tot(costs_iJO1366_reduced, medium_iJO1366_reduced).cpu().numpy().item()

print(init_growth, init_cost, sep = "\n")

0.8217947382570283
1239.59670934


## Sensitivity analysis
-> **Supplementary Figure 6 (iML1515 n_iter = 200, n_candidates = 20)**

Computing both objectives from N random compositions sampled from a multivariate normal centred at each point in the Pareto front and with an increasing variance. 

We generate 3 versions of the perturbed Pareto, with a cloud of objective values around it; for coefficient of variation = 5%, 10% and 25% of the mean.

### Prepare Data

In [ ]:
file_name = "Results\\2026-03-31_BayesOpt_iML1515_growth-cost_200.json"
results_iML1515_200 = JSON_deserialize_load_results((file_name), model_iML1515)

In [ ]:
traces_iML1515 = traces_M9_medium

In [30]:
# medium compositions
media_df = pd.DataFrame(results_iML1515_200["medium list"])
media_var_df = media_df.drop(list(traces_iML1515.keys()), axis = 1)
decision_vars_iML1515 = media_var_df.columns.tolist()


# growth and production
growth_np = results_iML1515_200["growth rate tensors"].cpu().numpy()
cost_np = results_iML1515_200["cost tensors"].cpu().numpy()
pareto_np = results_iML1515_200["is pareto"].cpu().numpy()

iML1515_200_df = pd.DataFrame({
    "growth_rate": growth_np,
    "cost" : cost_np,
    "is_pareto" : pareto_np,
    "init_growth_rate": init_growth,
    "init_cost": init_cost,
})
# Save to CSV
iML1515_200_df.to_csv("SuppFigure6_iML1515_200.csv", index = False)

pareto_data_indexes = iML1515_200_df[iML1515_200_df["is_pareto"] == True].index

"""
Cross-reference medium composition data with growth and cost data to extract the rows corresponding to the Pareto points 
"""
# extract the rows corresponding to the Pareto points from the medium data
sensitivity_analysis_medium_data_iML1515_200 = media_var_df.iloc[pareto_data_indexes]

In [32]:
"""Save random sampling data"""

sensitivity_analysis_medium_data_iML1515_200.to_csv("SuppFigure6_for-sampling_iML1515_200.csv", index = False)

### Generate random sampling for 3 different variances 
- 5% 
- 10%
- 25% 

In [ ]:
sample_dict_5percent_variance = generate_samples_from_multivariate_gaussian(
    sensitivity_analysis_medium_data_iML1515_200, 
    variance = 0.05, 
    sample_size = 1000, 
    dec_var_names = decision_vars_iML1515,
    bounds = bounds_iJO1366_reduced)

growth_and_cost_5_df = get_growth_and_cost(
    sample_dict_5percent_variance, 
    MetModel = model_iML1515, 
    costs = costs_iJO1366_reduced,
    production_rxn_id = None,
    traces = traces_iML1515)

#growth_and_cost_5_df

In [ ]:
sample_dict_10percent_variance = generate_samples_from_multivariate_gaussian(
    sensitivity_analysis_medium_data_iML1515_200, 
    variance = 0.1, 
    sample_size = 1000, 
    dec_var_names = decision_vars_iML1515,
    bounds = bounds_iJO1366_reduced)

growth_and_cost_10_df = get_growth_and_cost(
    sample_dict_10percent_variance, 
    MetModel = model_iML1515, 
    costs = costs_iJO1366_reduced,
    production_rxn_id = None,
    traces = traces_iML1515)

# growth_and_cost_10_df

In [ ]:
sample_dict_25percent_variance = generate_samples_from_multivariate_gaussian(
    sensitivity_analysis_medium_data_iML1515_200, 
    variance = 0.25, 
    sample_size = 1000, 
    dec_var_names = decision_vars_iML1515,
    bounds = bounds_iJO1366_reduced)

growth_and_cost_25_df = get_growth_and_cost(
    sample_dict_25percent_variance, 
    MetModel = model_iML1515, 
    costs = costs_iJO1366_reduced,
    production_rxn_id = None,
    traces = traces_iML1515)

#growth_and_cost_25_df

In [30]:
"""Save growth and cost dataframes for every variance level"""

growth_and_cost_5_df.to_csv("SuppFigure8_sensitivity_analysis_iML1515_200_5percent_variance.csv", index = False)
growth_and_cost_10_df.to_csv("SuppFigure8_sensitivity_analysis_iML1515_200_10percent_variance.csv", index = False)
growth_and_cost_25_df.to_csv("SuppFigure8_sensitivity_analysis_iML1515_200_25percent_variance.csv", index = False)